In [2]:
pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [keras-tuner]


In [3]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [4]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_val, y_val = x_train[-10000:], y_train[-10000:]
x_train, y_train = x_train[:-10000], y_train[:-10000]

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
def build_model(hp):

    model = keras.Sequential()
    model.add(layers.Flatten(input_shape=(28, 28)))

    # Tune the number of hidden layers
    num_layers = hp.Int("num_layers", min_value=1, max_value=3)

    for i in range(num_layers):

        # Tune the number of units
        units = hp.Int(
            f"units_{i}",
            min_value=32,
            max_value=256,
            step=32
        )

        model.add(layers.Dense(units, activation="relu"))

        # Tune dropout
        dropout_rate = hp.Float(
            f"dropout_{i}",
            min_value=0.0,
            max_value=0.5,
            step=0.1
        )

        model.add(layers.Dropout(dropout_rate))

    # Tune learning rate
    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.001, 0.0005, 0.0001]
    )

    model.add(layers.Dense(10, activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [6]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=1,
    directory="mnist_tuning",
    project_name="dense_network"
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
tuner.search(
    x_train,
    y_train,
    epochs=5,
    validation_data=(x_val, y_val)
)

Trial 10 Complete [00h 01m 19s]
val_accuracy: 0.9699000120162964

Best val_accuracy So Far: 0.9757000207901001
Total elapsed time: 00h 08m 06s


In [8]:
best_models = tuner.get_best_models(num_models=1)

best_model = best_models[0]

best_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 192)            │       150,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 192)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 96)             │        18,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 182,954 (714.66 KB)

 Trainable params: 182,954 (714.66 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best number of layers:", best_hps.get("num_layers"))
print("Best learning rate:", best_hps.get("learning_rate"))

Best number of layers: 3
Best learning rate: 0.0005


In [10]:
tuner.results_summary()

Results summary
Results in mnist_tuning/dense_network
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 07 summary
Hyperparameters:
num_layers: 3
units_0: 192
dropout_0: 0.4
learning_rate: 0.0005
units_1: 96
dropout_1: 0.1
units_2: 128
dropout_2: 0.1
Score: 0.9757000207901001

Trial 05 summary
Hyperparameters:
num_layers: 2
units_0: 64
dropout_0: 0.1
learning_rate: 0.0005
units_1: 128
dropout_1: 0.2
units_2: 256
dropout_2: 0.4
Score: 0.974399983882904

Trial 04 summary
Hyperparameters:
num_layers: 1
units_0: 128
dropout_0: 0.0
learning_rate: 0.0005
units_1: 256
dropout_1: 0.0
units_2: 64
dropout_2: 0.30000000000000004
Score: 0.9726999998092651

Trial 01 summary
Hyperparameters:
num_layers: 3
units_0: 128
dropout_0: 0.4
learning_rate: 0.001
units_1: 32
dropout_1: 0.0
units_2: 32
dropout_2: 0.0
Score: 0.9713000059127808

Trial 09 summary
Hyperparameters:
num_layers: 3
units_0: 192
dropout_0: 0.1
learning_rate: 0.0001
units_1: 160
dropout_1: 0.0
units_2: 160
dr

In [11]:
test_loss, test_accuracy = best_model.evaluate(x_test, y_test)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9756 - loss: 0.0845
Test Accuracy: 0.975600004196167
Test Loss: 0.08448176831007004
